# GastroVision — Baseline (shared base notebook)

**AIN501 Final Project**

This notebook is the **shared foundation** for the whole team. It establishes:
1. Reproducible config + seeding
2. Dataset loading with a **fixed stratified 60:20:20 split** (no ad-hoc re-splitting -> no leakage)
3. A **shared evaluation harness** (macro-F1, per-class F1, confusion matrix, multi-seed runner)
4. The **DenseNet-121 baseline** to reproduce (target macro-F1 ~0.65)

> Do NOT change the split / seed / eval cells without telling the team — everyone's
> numbers must be comparable.

**Target runtime: Google Colab Free (T4).** One run ~15-25 min @224px. Also works on Kaggle.

## 1 · Setup & reproducibility

In [ ]:
# Colab already ships torch/torchvision/sklearn/pandas/matplotlib/seaborn.
# `timm` (needed for Swin/ViT/ConvNeXt/CoAtNet) is NOT preinstalled:
!pip install -q timm

import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torchvision as tv
from torchvision import transforms
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| torch:', torch.__version__)
if DEVICE == 'cpu':
    print('WARNING: no GPU. On Colab: Runtime > Change runtime type > T4 GPU.')

In [ ]:
def set_seed(seed: int):
    """Full reproducibility. We report mean +/- std over >=3 seeds."""
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEEDS = [0, 1, 2]        # >=3 seeds required
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 30              # baseline; tune per experiment
NUM_WORKERS = 2
PIN_MEMORY = (DEVICE == 'cuda')   # pinning only helps host->GPU copies; avoids a warning on CPU
set_seed(SEEDS[0])

## 2 · Get the data (auto-detects Colab vs local)

The next cell **detects the environment** and fetches the dataset accordingly — no manual steps on Colab:
- **Colab:** downloads the official **`Gastrovision.zip`** (~1.8 GB) with `gdown` (by file id, first
  run only), unzips into `/content`, and saves checkpoints to your Drive (they survive disconnects).
- **Local:** expects the dataset unzipped under the repo's `data/gastrovision/`; checkpoints go to `checkpoints/`.

The class folders are then discovered by a **recursive walk** (`scan_class_folders`, next section):
every directory that directly holds images is treated as a class, so an extra nesting level or
grouped **Upper-GI / Lower-GI** parent folders are handled automatically — no manual paths. The
compute device (GPU/CPU) was already auto-detected in section 1 — no change needed to run in either place.

In [ ]:
# --- Environment-aware paths + SELF-HEALING auto-download (Colab AND local). ---
import os, subprocess

IMG_EXT_DL = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
def count_images(root):
    root = Path(root)
    if not root.exists():
        return 0
    return sum(1 for r, _, fs in os.walk(root)
               for f in fs if os.path.splitext(f)[1].lower() in IMG_EXT_DL)

def extract_all_zips(base):
    """Extract every .zip under base, repeatedly — the official archive is a zip-inside-a-zip
    (Gastrovision.zip -> Gastrovision.zip -> class folders). Stops when images appear or no zip left."""
    for _ in range(6):
        if count_images(base) > 1000:
            break
        inner = list(Path(base).rglob('*.zip'))
        if not inner:
            break
        for z in inner:
            print('extracting nested zip:', z.name)
            subprocess.run(['unzip', '-q', '-o', str(z), '-d', str(z.parent)], check=True)
            z.unlink()          # remove so it isn't re-extracted next loop

def print_tree(root, max_depth=2):
    root = Path(root)
    print('--- tree of', root, '---')
    for r, dirs, files in os.walk(root):
        depth = r[len(str(root)):].count(os.sep)
        if depth <= max_depth:
            print('  ' * depth, os.path.basename(r) or str(root), f'/ ({len(files)} files)')
        if depth >= max_depth:
            dirs[:] = []

try:
    from google.colab import drive
    IS_COLAB = True
except ModuleNotFoundError:
    IS_COLAB = False
print('environment:', 'Colab' if IS_COLAB else 'local')

# Official GastroVision zip (Gastrovision.zip, ~1.8 GB). File id from the author's Drive folder
# https://drive.google.com/drive/folders/1oT9Vez7pfhrN44Korx6wHRBAGwyHc7Cb
GDRIVE_ZIP_ID = '1VV-gW0PqtykFfoA-1BswH8nYy9JByuxU'

if IS_COLAB:
    BASE       = Path('/content/gastrovision')
    OUTPUT_DIR = Path('/content/outputs')
    drive.mount('/content/drive')                                  # for checkpoint persistence
    CKPT_DIR   = Path('/content/drive/MyDrive/gastrovision_ckpts')
    BASE.mkdir(parents=True, exist_ok=True)

    # Gate on the number of IMAGES actually present, NOT on the folder existing — so a leftover
    # empty (or zip-only) /content/gastrovision from a failed run does not block re-extraction.
    n_have = count_images(BASE)
    print('images currently under BASE:', n_have)
    if n_have < 1000:
        subprocess.run(['pip', 'install', '-q', '-U', 'gdown'], check=True)
        zip_path = Path('/content/Gastrovision.zip')
        # (Re)download if the zip is missing or clearly too small (a Drive virus-scan HTML page
        # is only a few KB — a valid download is ~1.8 GB).
        if (not zip_path.exists()) or zip_path.stat().st_size < 500_000_000:
            print('downloading Gastrovision.zip (~1.8 GB, first run only)...')
            subprocess.run(['gdown', f'https://drive.google.com/uc?id={GDRIVE_ZIP_ID}',
                            '-O', str(zip_path)], check=True)
        sz_mb = zip_path.stat().st_size / 1e6
        print(f'zip size: {sz_mb:.0f} MB')
        assert zip_path.stat().st_size > 500_000_000, (
            f'Gastrovision.zip is only {sz_mb:.1f} MB -> gdown likely returned the Drive '
            'virus-scan page, not the file. Delete /content/Gastrovision.zip and retry, or '
            'download manually from OSF: https://osf.io/84e7f/')
        print('unzipping (archive is a zip-inside-a-zip)...')
        subprocess.run(['unzip', '-q', '-o', str(zip_path), '-d', str(BASE)], check=True)
        extract_all_zips(BASE)          # peel the nested Gastrovision.zip -> real class folders
        print('images after unzip:', count_images(BASE))
else:
    REPO       = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    BASE       = REPO / 'data' / 'gastrovision'   # unzip Gastrovision.zip here
    CKPT_DIR   = REPO / 'checkpoints'
    OUTPUT_DIR = REPO / 'outputs'
    extract_all_zips(BASE)              # local: handle a nested zip too, if present

CKPT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert BASE.exists(), f'Dataset base not found at {BASE}. Colab: check gdown. Local: unzip into {BASE}.'
assert count_images(BASE) > 1000, (
    f'Only {count_images(BASE)} images under {BASE} -> extraction did not reach the images. '
    'See the tree below. On Colab, delete /content/gastrovision AND /content/Gastrovision.zip, '
    'then re-run this cell to start clean.')

# Class folders are discovered RECURSIVELY in the next cell (see scan_class_folders), so an
# extra nesting level or grouped Upper-GI/Lower-GI parent folders are handled automatically.
DATA_DIR = BASE
print('\nDATA_DIR   =', DATA_DIR, '| total images:', count_images(DATA_DIR))
print('CKPT_DIR   =', CKPT_DIR, '\n')
print_tree(DATA_DIR, max_depth=2)

### 2.1 · Verify the download, then filter to the 22 experiment classes

The previous cell downloads the **official single zip** `Gastrovision.zip` (~1.8 GB). The next cell
**recursively walks** the extracted tree (`scan_class_folders`) and treats every folder that
directly contains images as a class — this handles GastroVision's nested **Upper-GI / Lower-GI**
category folders that `ImageFolder` (single-level only) cannot. Because it's one zip (not thousands
of loose files), the download is complete and reliable — no `gdown --folder` 50-file cap to worry about.

**How you know the data is complete:** the next cell prints a **download-integrity report** —
total images, class-folder count, full per-class distribution, and a corrupt-file spot-check —
then **hard-fails if fewer than ~8,000 images are present**. If it passes, the data is complete.

After the integrity check, it filters to the paper's **22 experiment classes**. The paper
(arXiv 2307.08140) states verbatim: *"we have only included classes with more than 25 samples in
the experiments, which resulted in 22 classes in total."* We apply the same **`count > 25`** rule
and `assert NUM_CLASSES == 22`.

**The 22 experiment classes** (numbers = **full-set** image counts, verified from this download).
The full set has 27 classes; the **5 rarest are dropped** (≤25 images):
`Resection margins (25), Angiectasia (17), Erythema (15), Esophageal varices (7), Ulcer (6)`.

- **Upper GI:** Barrett's esophagus (95), Blood in lumen (171), Duodenal bulb (205),
  Esophagitis (107), Gastric polyps (65), Gastroesophageal junction / normal z-line (330),
  Normal esophagus (140), Normal stomach (969), Pylorus (393)
- **Lower GI:** Accessory tools (1266), Cecum (113), Colon diverticula (29), Colon polyps (820),
  Colorectal cancer (139), Dyed-lifted-polyps (141), Dyed-resection-margins (246), Ileocecal valve (200),
  Mucosal inflammation large bowel (29), Normal mucosa & vascular pattern large bowel (1467),
  Resected polyps (92), Retroflex rectum (67), Small bowel / terminal ileum (846)

> The imbalance is severe — **1467 vs 29** among the kept classes (≈50×). This long tail is exactly
> why data-centric work (class-balancing + macro-F1) matters, and why the baseline macro-F1 is only 0.6504.

In [ ]:
import os
from PIL import Image

# GastroVision nests its class folders under anatomical-category parents (e.g. Upper GI / Lower GI),
# so torchvision.ImageFolder (which reads exactly ONE level of class folders) fails with
# "Couldn't find any class folder". Instead we WALK the whole tree and treat every directory
# that DIRECTLY contains image files as a class (class name = that folder's basename).
# This is robust to any nesting depth and to grouped category folders.
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
SKIP_DIRS = {'__MACOSX', 'Source', 'Split', 'source', 'split'}   # code/split folders, not classes

def scan_class_folders(root):
    """Return {class_name: [image_path, ...]}. A class folder = any dir directly holding >=1 image."""
    by_class = {}
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if not d.startswith('.') and d not in SKIP_DIRS]
        imgs = [f for f in filenames if os.path.splitext(f)[1].lower() in IMG_EXT]
        if not imgs:
            continue
        cls = os.path.basename(dirpath.rstrip('/'))
        by_class.setdefault(cls, []).extend(os.path.join(dirpath, f) for f in imgs)
    return by_class

by_class = scan_class_folders(str(DATA_DIR))
assert by_class, f'No image folders found under {DATA_DIR}. Check the download/unzip cell above.'

ORIG_CLASSES = sorted(by_class)
cls_to_idx = {c: i for i, c in enumerate(ORIG_CLASSES)}
all_samples = [(p, cls_to_idx[c]) for c, paths in by_class.items() for p in paths]  # (path, class_idx)
all_labels = np.array([y for _, y in all_samples])

# ===== 2.2 - Download-integrity report (confirm the full ~8k dataset is present) =====
EXPECTED_TOTAL = 8000   # documented in the paper / GitHub README (27 classes)
per_class = pd.Series(all_labels).value_counts().sort_values(ascending=False)
per_class.index = [ORIG_CLASSES[i] for i in per_class.index]
print(f'total images  : {len(all_samples)}  (expected ~{EXPECTED_TOTAL})')
print(f'class folders : {len(ORIG_CLASSES)}  (expected 27)')
print('\nper-class counts (full set, all folders):')
print(per_class.to_string())

bad = 0                                 # spot-check that images actually open (catch partial files)
for p, _ in all_samples[::50]:
    try:
        Image.open(p).verify()
    except Exception:
        bad += 1
print(f'\ncorrupt images in sample (every 50th file): {bad}')

assert len(all_samples) >= 0.9 * EXPECTED_TOTAL, (
    f'Only {len(all_samples)}/{EXPECTED_TOTAL} images -> dataset looks INCOMPLETE. '
    'Re-run the download cell, or grab the zip from OSF: https://osf.io/84e7f/')
assert bad == 0, f'{bad} corrupt/partial images in sample -> the zip may be truncated; re-download.'
print('\nOK - dataset looks complete.')

# ===== Filter to the 22 experiment classes =====
# Paper (arXiv 2307.08140), verbatim: "we have only included classes with more than 25 samples
# in the experiments, which resulted in 22 classes in total." -> keep count > 25 (i.e. >= 26).
# This drops the 5 rarest: Resection margins(25), Angiectasia(17), Erythema(15),
# Esophageal varices(7), Ulcer(6) -> exactly the paper's 22-class experimental subset.
MIN_PER_CLASS = 26   # ">25 samples"
counts_orig = np.bincount(all_labels, minlength=len(ORIG_CLASSES))
keep = [c for c in range(len(ORIG_CLASSES)) if counts_orig[c] >= MIN_PER_CLASS]
remap = {c: i for i, c in enumerate(keep)}

CLASSES = [ORIG_CLASSES[c] for c in keep]
NUM_CLASSES = len(CLASSES)
samples = [(p, remap[y]) for (p, y) in all_samples if y in remap]   # (path, remapped 0..21 idx)
labels = np.array([y for _, y in samples])

dropped = [f'{ORIG_CLASSES[c]}({counts_orig[c]})'
           for c in range(len(ORIG_CLASSES)) if c not in remap]
print(f'\nkept {NUM_CLASSES}/{len(ORIG_CLASSES)} classes for experiments, {len(samples)} images')
if dropped:
    print(f'dropped (<={MIN_PER_CLASS - 1} imgs): {dropped}')

# If this fails, the scan picked up unexpected folders. The per-class print + the top-level
# listing from the previous cell show exactly what was found.
assert NUM_CLASSES == 22, (
    f'Expected 22 classes (paper), got {NUM_CLASSES}. Found folders: {ORIG_CLASSES}')

In [ ]:
# --- Fixed stratified 60:20:20 split (paper protocol). Keep EXACT across the team. ---
SPLIT_SEED = 42
idx = np.arange(len(samples))
train_idx, tmp_idx = train_test_split(
    idx, test_size=0.40, stratify=labels, random_state=SPLIT_SEED)
val_idx, test_idx = train_test_split(
    tmp_idx, test_size=0.50, stratify=labels[tmp_idx], random_state=SPLIT_SEED)
print(f'train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}')

# NOTE: SPLIT_SEED is fixed so the split is identical for everyone. The per-run SEEDS only
# affect model init / data ordering, NOT the split.

## 3 · EDA — long-tail check (shared Data work)

The data-analysis section. Below is a starter; expand together with image-quality
checks, artifacts, suspicious labels, and a **leakage check**.

In [ ]:
import matplotlib.pyplot as plt
counts = pd.Series(labels).value_counts().sort_index()
counts.index = [CLASSES[i] for i in counts.index]
counts = counts.sort_values(ascending=False)
print('most/least common:')
print(counts.head(3)); print(counts.tail(3))
print(f'imbalance ratio (max/min): {counts.max() / counts.min():.1f}x')

plt.figure(figsize=(12, 4))
counts.plot(kind='bar'); plt.ylabel('# images')
plt.title('GastroVision class distribution (long-tail)')
plt.tight_layout(); plt.show()
# TODO: flag classes with few images -> macro-F1 will be noisy there.

## 4 · Datasets & transforms

> WARNING - **Team (shared Data work):** the augmentation here is a placeholder. Agree on ONE
> **GI-domain-justified** recipe and prove each step with an ablation. This same recipe is
> used by all three architectures so their numbers are comparable.

In [ ]:
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),          # TODO: justify per GI domain
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class GastroDataset(Dataset):
    def __init__(self, indices, transform):
        self.items = [samples[i] for i in indices]
        self.transform = transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, y = self.items[i]
        img = Image.open(path).convert('RGB')    # some GI frames may be grayscale/RGBA
        return self.transform(img), y

def make_loaders():
    tr = DataLoader(GastroDataset(train_idx, train_tf), batch_size=BATCH_SIZE,
                    shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    va = DataLoader(GastroDataset(val_idx, eval_tf), batch_size=BATCH_SIZE,
                    shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    te = DataLoader(GastroDataset(test_idx, eval_tf), batch_size=BATCH_SIZE,
                    shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    return tr, va, te

## 5 · Shared evaluation harness (everyone imports)

**Primary metric = macro-F1.** Always report per-class F1 + confusion matrix.
`labels=range(NUM_CLASSES)` is passed everywhere so a rare class missing from predictions
never crashes the report.

In [ ]:
ALL_LABELS = list(range(NUM_CLASSES))

@torch.no_grad()
def evaluate(model, loader):
    """Return dict with macro_f1, micro_f1, y_true, y_pred."""
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        x = x.to(DEVICE)
        logits = model(x)
        ps.append(logits.argmax(1).cpu().numpy())
        ys.append(y.numpy())
    y_true = np.concatenate(ys); y_pred = np.concatenate(ps)
    return {
        'macro_f1': f1_score(y_true, y_pred, labels=ALL_LABELS, average='macro', zero_division=0),
        'micro_f1': f1_score(y_true, y_pred, labels=ALL_LABELS, average='micro', zero_division=0),
        'y_true': y_true, 'y_pred': y_pred,
    }

def report_per_class(res):
    print(classification_report(res['y_true'], res['y_pred'], labels=ALL_LABELS,
          target_names=CLASSES, zero_division=0, digits=3))

def plot_confusion(res):
    import seaborn as sns
    cm = confusion_matrix(res['y_true'], res['y_pred'], labels=ALL_LABELS)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES, cbar=False)
    plt.xlabel('pred'); plt.ylabel('true'); plt.tight_layout(); plt.show()

## 6 · Baseline model — DenseNet-121 (reproduce first!)

**Reproduce the strongest published baseline before claiming to beat it.**
Target: land near the paper's macro-F1 **0.6504**. Report YOUR reproduced number.

In [ ]:
def build_densenet121(num_classes):
    m = tv.models.densenet121(weights=tv.models.DenseNet121_Weights.IMAGENET1K_V1)
    m.classifier = nn.Linear(m.classifier.in_features, num_classes)
    return m.to(DEVICE)

USE_AMP = (DEVICE == 'cuda')

def train_one(model, tr, va, epochs=EPOCHS, lr=1e-4, class_weights=None):
    """Minimal fine-tune loop w/ AMP + best-val checkpointing. Returns best val macro-F1."""
    crit = nn.CrossEntropyLoss(weight=class_weights)   # TODO: swap for LDAM/Balanced-Softmax
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    best, best_state = -1.0, None
    for ep in range(epochs):
        model.train()
        for x, y in tr:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        vf1 = evaluate(model, va)['macro_f1']
        if vf1 > best:
            best, best_state = vf1, {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f'ep {ep+1:02d}/{epochs}  val_macroF1={vf1:.4f}  (best={best:.4f})')
    if best_state:
        model.load_state_dict(best_state)
    return best

In [ ]:
# --- Multi-seed runner: report mean +/- std on the TEST split. ---
def run_seeds(build_fn, seeds=SEEDS, tag='model', **train_kw):
    test_scores = []
    for s in seeds:
        set_seed(s)
        tr, va, te = make_loaders()
        model = build_fn(NUM_CLASSES)
        train_one(model, tr, va, **train_kw)
        res = evaluate(model, te)
        print(f'[seed {s}] test macro-F1 = {res["macro_f1"]:.4f}')
        test_scores.append(res['macro_f1'])
        torch.save(model.state_dict(), CKPT_DIR / f'{tag}_seed{s}.pt')   # survives disconnects
    mean, std = float(np.mean(test_scores)), float(np.std(test_scores))
    print(f'\n{tag} TEST macro-F1: {mean:.4f} +/- {std:.4f}')
    return {'tag': tag, 'scores': test_scores, 'mean': mean, 'std': std}

# Reproduce baseline: start with 1 seed to sanity-check ~0.65, then run all 3 seeds.
# baseline = run_seeds(build_densenet121, seeds=[0], tag='densenet121')   # quick check
# baseline = run_seeds(build_densenet121, tag='densenet121')              # full 3-seed

---
## 7 · Team plan (branch from here)

**Data work is done together** (one shared recipe so all numbers are consistent); the **Model
work is one baseline per person.** Everyone reuses `run_seeds` / `evaluate` so results are comparable.

### Shared by all — Data
Agree on ONE data recipe applied to every architecture:
- GI-domain augmentation + **ablation table** (each step on/off -> macro-F1).
- Imbalance handling: class weights -> LDAM / Balanced-Softmax / decoupled (cRT) / repeat-factor sampling.
- EDA depth + leakage check + suspicious-label review.

### One baseline per person — Model
Each member owns one architecture family (this gives the CNN vs Transformer vs Hybrid comparison).
Each: build -> `run_seeds(build_fn, tag=...)` on the fixed split + shared data recipe.

- **Member A — CNN:** DenseNet-121 (`build_densenet121`, above) — reproduce the 0.6504 baseline first.
- **Member B — Transformer:** Swin-T / ViT-S via `timm`.
- **Member C — Hybrid:** CoAtNet via `timm`.

```python
# import timm
# def build_swin_t(nc):  return timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=nc).to(DEVICE)
# def build_coatnet(nc): return timm.create_model('coatnet_0_rw_224', pretrained=True, num_classes=nc).to(DEVICE)
# run_seeds(build_swin_t, tag='swin_t')
```

### Shared responsibilities (divide as you like)
- **Eval harness** (macro-F1, per-class F1, confusion matrix, multi-seed) — already provided above.
- **Transfer learning:** linear probe vs progressive unfreezing vs layer-wise LR vs full fine-tune.
- **Deployment:** ONNX export + latency/size + Gradio demo.

```python
# dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
# torch.onnx.export(model, dummy, str(OUTPUT_DIR / 'model.onnx'),
#                   input_names=['input'], output_names=['logits'], opset_version=17)
```

**Final step:** combine -> pick the best `(architecture x data recipe)` as the proposed model and
compare it against the three reproduced baselines.